# Experimental Carvana sales assumptions and scenarios
This notebook separates retained observations, research assumptions and unavailable sales estimates.
Optional dated scenario analysis, not measured sales or the daily operating workflow.
The default historical cutoff selects no daily cycles automatically; short retained history cannot support a quarter-to-date sales forecast.
Daily collection is assembled over an interval. “Updated” means the displayed cutoff, not continuous monitoring.

Carvana includes reserved/purchase-in-progress and some pre-reconditioning vehicles in website units.
Retail units sold include marketplace partners and are net of returns under its seven-day policy.
Source: [Q2 2026 filing, Key Operating Metrics](https://www.sec.gov/Archives/edgar/data/1690820/000169082026000055/cvna-20260630.htm);
[results published July 29, 2026](https://investors.carvana.com/news-releases/2026/07-29-2026-210525685).
Publication date alone is not an intraday availability timestamp. No benchmark is populated automatically.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd()
if (ROOT / 'vehicle/src').is_dir(): ROOT = ROOT / 'vehicle'
elif ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from vehicle_tracker.cycles import read_cycle_history
from vehicle_tracker.events import vin_events, daily_counts
from vehicle_tracker.expectations import quarter_coverage, dated_input, revision_bridge, export_research
AS_OF = globals().get('AS_OF_OVERRIDE', '2026-09-08T13:00:00Z')
QUARTER = globals().get('QUARTER_OVERRIDE', '2026Q3')
TIMEZONE = 'America/New_York'
DATABASE = Path(globals().get('DATABASE_OVERRIDE', ROOT / 'data/analysis/carvana_daily/history.sqlite'))
CYCLE_REPORTS = globals().get('CYCLE_REPORTS_OVERRIDE', [])
days, source_rows = read_cycle_history(CYCLE_REPORTS, DATABASE, as_of=AS_OF)
calendar, research_status = quarter_coverage(days, as_of=AS_OF, quarter=QUARTER, timezone_name=TIMEZONE)
display(research_status)
display(calendar)
print('Read-only database:', DATABASE, 'No collection, import or export occurs by default.')

In [ ]:
daily_activity = pd.DataFrame()
if not days.empty:
    events = vin_events(days, source_rows, absence_days=3)
    daily_activity = daily_counts(days, events)
    quarter_activity = calendar.merge(daily_activity[['cycle_date', 'observed_vins', 'pending_started', 'pending_cleared', 'persistent_absence']], on='cycle_date', how='left', validate='one_to_one')
    display(quarter_activity)
    quarter_activity.assign(date=pd.to_datetime(quarter_activity.cycle_date)).set_index('date')[['observed_vins', 'pending_started']].plot(subplots=True, figsize=(10, 5), title='Observed activity: missing dates stay gaps')
    plt.tight_layout()
    plt.show()
else:
    print('NO DAILY EVIDENCE. Historical intraday observations cannot fill missing quarter dates.')
print('Actual sales estimate:', research_status.estimated_retail_units.iloc[0])

## Explicit analyst inputs
No conversion, daily run rate or consensus is supplied. A dated input requires an ID, source,
availability time, quarter, population, metric, units and value. Select a version explicitly.
Missing, stale, future or mismatched inputs block comparison. A populated assumption is not
evidence of calibrated accuracy. Validation requires subsequent observed history and reviewed labels.

The arithmetic below is a **synthetic teaching example**, not a Carvana forecast. It illustrates an
assumed conversion from observed activity and an assumed rate for remaining days. Those quantities
would need population coverage and prospective validation before investment use.

In [ ]:
ANALYST_INPUTS = globals().get('ANALYST_INPUTS_OVERRIDE', [])
BENCHMARK = globals().get('BENCHMARK_OVERRIDE')
display(pd.DataFrame(ANALYST_INPUTS, columns=['input_id', 'source', 'available_at', 'quarter', 'scope_id', 'metric', 'units', 'value']))
benchmark_value = None
if BENCHMARK is not None:
    try:
        benchmark_value = dated_input(BENCHMARK, as_of=AS_OF, quarter=QUARTER,
            scope_id=research_status.scope_id.iloc[0], metric='retail_units', units='vehicles', max_age_days=7)
        print('Compatible benchmark:', benchmark_value, 'No spread calculated without a sales estimate.')
    except ValueError as error:
        print('BENCHMARK BLOCKED:', error)
else:
    print('No dated guidance/consensus selected. No benchmark or surprise is invented.')
analyst_scenario = pd.DataFrame()
if ANALYST_INPUTS:
    try:
        if not calendar.day_status.eq('complete').all() or research_status.age_hours.isna().any() or research_status.age_hours.iloc[0] > 36:
            raise ValueError('Complete fresh quarter dates are required; do not fill gaps with zero')
        inputs = pd.DataFrame(ANALYST_INPUTS).set_index('metric')
        if not inputs.index.is_unique: raise ValueError('Select one explicit version per input metric')
        conversion_record = inputs.loc['absence_conversion'].to_dict() | {'metric':'absence_conversion'}
        rate_record = inputs.loc['remaining_daily_units'].to_dict() | {'metric':'remaining_daily_units'}
        scope_id = research_status.scope_id.iloc[0]
        assumed_conversion = dated_input(conversion_record, as_of=AS_OF, quarter=QUARTER, scope_id=scope_id, metric='absence_conversion', units='units_per_absence', max_age_days=7)
        assumed_daily_rate = dated_input(rate_record, as_of=AS_OF, quarter=QUARTER, scope_id=scope_id, metric='remaining_daily_units', units='vehicles_per_day', max_age_days=7)
        if assumed_conversion > 1: raise ValueError('An absence conversion must be between zero and one')
        covered_activity = daily_activity[daily_activity.cycle_date.isin(calendar.cycle_date)].persistent_absence.sum(min_count=1)
        assumed_units_through_cutoff = covered_activity * assumed_conversion
        assumed_remaining_units = research_status.remaining_days.iloc[0] * assumed_daily_rate
        analyst_scenario = pd.DataFrame([dict(as_of=AS_OF, quarter=QUARTER, status='EXPLORATORY ASSUMPTION SCENARIO',
            activity=covered_activity, assumed_units_through_cutoff=assumed_units_through_cutoff,
            assumed_remaining_units=assumed_remaining_units, scenario_units=assumed_units_through_cutoff+assumed_remaining_units)])
        display(analyst_scenario)
        print('Scoped assumption scenario, not validated national sales. Actual sales estimate remains unavailable.')
    except (ValueError, KeyError) as error:
        print('ANALYST SCENARIO BLOCKED:', error)

In [ ]:
print('SYNTHETIC SCENARIO ONLY; not a calibrated Carvana estimate.')
example_source = dict(input_id='synthetic-conversion-v1', source='synthetic://teaching', available_at='2026-09-01T00:00:00Z', quarter='2026Q3', scope_id='synthetic', metric='conversion', units='units_per_activity', value=0.5)
conversion = dated_input(example_source, as_of='2026-09-01T23:59:00Z', quarter='2026Q3', scope_id='synthetic', metric='conversion', units='units_per_activity')
prior_daily = pd.DataFrame({'date':['2026-07-01', '2026-09-01'], 'activity':[6., 4.]})
current_daily = pd.DataFrame({'date':['2026-07-01', '2026-09-01', '2026-09-02'], 'activity':[6., 5., 4.]})
# Compressed teaching inputs, not complete observed quarter calendars.
comparison = prior_daily.merge(current_daily, on='date', how='outer', suffixes=('_previous', '_current'), indicator=True, validate='one_to_one')
display(comparison)
previous = dict(as_of='2026-09-01T23:59:00Z', quarter='2026Q3', scope_id='synthetic', metric='scenario_units', units='vehicles', coverage_basis='synthetic assumed complete quarter inputs', activity=float(prior_daily.activity.sum()), conversion=conversion, remaining_days=29, daily_rate=2.)
current = dict(previous, as_of='2026-09-02T23:59:00Z', activity=float(current_daily.activity.sum()), conversion=0.6, remaining_days=28, daily_rate=3.)
prior_scenario = previous['activity']*previous['conversion'] + previous['remaining_days']*previous['daily_rate']
current_scenario = current['activity']*current['conversion'] + current['remaining_days']*current['daily_rate']
revised_common_activity = float(comparison.loc[comparison['_merge'].eq('both'), 'activity_current'].sum())
bridge = revision_bridge(previous, current, prior_activity_revised=revised_common_activity)
display(bridge)
display(pd.DataFrame([dict(previous_scenario=prior_scenario, current_scenario=current_scenario, revision=current_scenario-prior_scenario, bridge_residual=current_scenario-prior_scenario-bridge.scenario_unit_change.sum())]))
try:
    revision_bridge(previous, dict(current, coverage_basis='partial coverage'), prior_activity_revised=revised_common_activity)
except ValueError as error:
    print('COVERAGE CHANGE BLOCKS ATTRIBUTION:', error)

## What would change a sales expectation?
Keep new observations, revisions to previously observed dates, passage of time and changed analyst
assumptions separate. The bridge uses that declared order; interactions depend on the order.
Incompatible coverage/populations block numeric attribution rather than being called sales acceleration.

Review 2/3/7-day absence sensitivity in notebook 20. Pending clearance is not a cancellation; a returning
listing is not proof of a returned purchase. Cars can appear and disappear between observations.
Prospective calibration needs comparable complete cycles, reviewed event evidence, marketplace/return
definitions and a later holdout period. A quarterly match cannot establish correct daily timing.
Asking prices do not establish transaction prices, revenue or earnings.

In [ ]:
EXPORT_DIRECTORY = globals().get('EXPORT_DIRECTORY_OVERRIDE')  # Explicit NEW directory only.
if EXPORT_DIRECTORY is not None:
    provenance_paths = [Path(path) for path in CYCLE_REPORTS]
    provenance_paths += [ROOT / 'notebooks/30_carvana_sales_expectations.ipynb']
    if DATABASE.is_file(): provenance_paths.append(DATABASE)
    from vehicle_tracker.cycles import cycle_evidence
    for path in CYCLE_REPORTS:
        _, _, reports = cycle_evidence(path, as_of=AS_OF)
        provenance_paths += reports
        for report in reports:
            provenance_paths += [Path(page['retained_source']) for page in json.loads(report.read_text())['pages']]
    export_research({'quarter_coverage':calendar, 'research_status':research_status, 'observed_activity':daily_activity, 'analyst_scenario':analyst_scenario},
        destination=EXPORT_DIRECTORY, as_of=AS_OF, source_paths=provenance_paths,
        assumptions={'quarter':QUARTER, 'timezone':TIMEZONE, 'absence_days':3, 'max_age_hours':36, 'analyst_inputs':ANALYST_INPUTS, 'benchmark':BENCHMARK})
else:
    print('Export disabled. A future explicit export records source/code/notebook hashes and assumptions.')